# Convertendo os IPTUS de São Paulo para formato HDF5

Com a intenção de ter mais eficiência e poder utilizar os recursos da biblioteca Vaex, assim como de analisar todo o período de IPTU disponibilizados pela Prefeitura de São Paulo, esse notebook documenta o processo de conversão do formato CSV original para HDF5

In [16]:
import vaex
import pandas as pd
# import numpy as np
import glob
import csv
import os
import zipfile
import numpy as np

## Definindo os tipos/formatos para cada coluna

In [17]:
dtypes = {'NUMERO DO CONTRIBUINTE': object,
        'ANO DO EXERCICIO': 'Int16', 
        'NUMERO DA NL': 'Int16',
        'CEP DO IMOVEL': object,
        'QUANTIDADE DE ESQUINAS/FRENTES': 'Float64',
        'FRACAO IDEAL': 'Float32',
        'AREA DO TERRENO': 'Int32',
        'AREA CONSTRUIDA': 'Int32',
        'AREA OCUPADA': 'Int32',
        'VALOR DO M2 DO TERRENO': 'Float32',
        'VALOR DO M2 DE CONSTRUCAO': 'Float32',
        'ANO DA CONSTRUCAO CORRIGIDO': 'Int16',
        'QUANTIDADE DE PAVIMENTOS': object, ## 'Int8', ##
        'TESTADA PARA CALCULO':  object, ## 'Float32', ##
        'TIPO DE USO DO IMOVEL': 'category',
        'TIPO DE PADRAO DA CONSTRUCAO': 'category',
        'TIPO DE TERRENO': 'category',
        'FATOR DE OBSOLESCENCIA': 'Float32',
        'ANO DE INICIO DA VIDA DO CONTRIBUINTE': 'Int16',
        'MES DE INICIO DA VIDA DO CONTRIBUINTE': 'Int8',
        'FASE DO CONTRIBUINTE': 'Int8'
}

In [18]:
exercicio = 2024

In [19]:
# print(f'Validando {exercicio} ...')

# try:
#     with open (f'data/IPTU_{exercicio}/IPTU_{exercicio}.csv', 'r') as f: 
#         linhas = f.readlines() 
# except:
#     print(f'Encoding iso-8859-9 IPTU {exercicio}')
#     with open (f'data/IPTU_{exercicio}/IPTU_{exercicio}.csv', 'r', encoding='iso-8859-9') as f: 
#         linhas = f.readlines() 
    
# linhas_inconformes = [i for i, x in enumerate(linhas) if x.count(';') != 28]

# if len(linhas_inconformes) > 0:
#         print(f' ************ {len(linhas_inconformes)} REGISTROS INCONFORMES ENCONTRADOS EM {exercicio}!')
        
#         registros_inconformes = [x for i, x in enumerate(linhas) if x.count(';') != 28]

In [20]:
# linhas_inconformes = [i for i, x in enumerate(linhas) if x.count(';') != 28]

In [21]:
# for i, x in enumerate(linhas):
#     if x.count(";") != 28:
#         print(i)

In [22]:
df_iptu_pd = pd.read_csv(
    f'data/IPTU_{exercicio}/IPTU_{exercicio}.csv', sep=';',
                    decimal=',',
                    encoding='iso-8859-9', on_bad_lines='skip',
                    dtype=dtypes)

In [23]:
df_iptu_pd.rename(columns={'QUANTIDADE DE ESQUINAS/FRENTES':'QUANTIDADE DE ESQUINAS FRENTES'}, inplace=True)

In [24]:
# qt_pavimentos_numeric = df_iptu_pd.loc[:, 'QUANTIDADE DE PAVIMENTOS'].str.isnumeric()

In [25]:
df_iptu_pd.loc[:, 'QUANTIDADE DE PAVIMENTOS'] = pd.to_numeric(df_iptu_pd.loc[:, 'QUANTIDADE DE PAVIMENTOS'], errors='coerce').astype('Int8').fillna(0)


In [26]:
df_iptu_pd.loc[:, 'TESTADA PARA CALCULO'] = pd.to_numeric(df_iptu_pd.loc[:, 'TESTADA PARA CALCULO'], errors='coerce').astype('Float32').fillna(0.0)


In [27]:
df_iptu_pd.dtypes

NUMERO DO CONTRIBUINTE                     object
ANO DO EXERCICIO                            Int16
NUMERO DA NL                                Int16
DATA DO CADASTRAMENTO                      object
NUMERO DO CONDOMINIO                       object
CODLOG DO IMOVEL                           object
NOME DE LOGRADOURO DO IMOVEL               object
NUMERO DO IMOVEL                          float64
COMPLEMENTO DO IMOVEL                      object
BAIRRO DO IMOVEL                           object
REFERENCIA DO IMOVEL                       object
CEP DO IMOVEL                              object
QUANTIDADE DE ESQUINAS FRENTES            Float64
FRACAO IDEAL                              Float32
AREA DO TERRENO                             Int32
AREA CONSTRUIDA                             Int32
AREA OCUPADA                                Int32
VALOR DO M2 DO TERRENO                    Float32
VALOR DO M2 DE CONSTRUCAO                 Float32
ANO DA CONSTRUCAO CORRIGIDO                 Int16


In [28]:
# df_iptu = vaex.from_csv(f'data/IPTU_{exercicio}/IPTU_{exercicio}.csv', sep=';',
#                     decimal=',',
#                     encoding='iso-8859-9', dtype=dtypes, on_bad_lines='skip')

In [29]:
df_iptu = vaex.from_pandas(df_iptu_pd)

In [30]:
df_iptu.export_hdf5(f'data/IPTU-HDF5/IPTU_{exercicio}/IPTU_{exercicio}.hdf5', progress=True)

export(hdf5) [########################################] 100.00% elapsed time  :     2.72s =  0.0m =  0.0h            
 